In [9]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql import functions as F
from itertools import chain

In [2]:
spark = SparkSession.builder \
    .appName("var") \
    .master("local[*]") \
    .config("spark.executor.memory", "512M") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/18 11:33:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


#### There are 2 types of variables:

* Distributed shared variables

* Broadcast variable: broadcast variable is a read-only, distributed shared variable that is cached on each executor node rather than being packaged and sent repeatedly with every parallel task

💡 Why Use Broadcast Variables?
* **Reduces Network I/O**: The data travels over the network only once per executor.

* **Minimizes Memory Overhead**: Multiple tasks running on the same executor share a single cached copy.

* **Optimizes Join Performance**: It allows Spark to perform Broadcast Joins, which merge small lookup dataframes with giant datasets completely without expensive data shuffling

In [3]:
_schema = """
    first_name string,
    last_name string,
    job_title string,
    dob string,
    email string,
    phone string,
    salary float,
    department_id int
"""
emp = spark.read.format("csv").schema(_schema).option("header", True).load("/Users/AnhHuynh/Documents/PySpark/employee_records.txt")

In [5]:
# Broadcast the varialbe

# Variable Lookup

dept_name = {1: "Department 1",
             2: "Department 2",
             3: "Department 3",
             4: "Department 4",
             5: "Department 5",
             6: "Department 6",
             7: "Department 7",
             8: "Department 8",
             9: "Department 9",
             10: "Department 10"
            }

broadcast_dept_names = spark.sparkContext.broadcast(dept_name)

In [6]:
type(broadcast_dept_names)

pyspark.core.broadcast.Broadcast

In [7]:
broadcast_dept_names.value

{1: 'Department 1',
 2: 'Department 2',
 3: 'Department 3',
 4: 'Department 4',
 5: 'Department 5',
 6: 'Department 6',
 7: 'Department 7',
 8: 'Department 8',
 9: 'Department 9',
 10: 'Department 10'}

In [22]:
mapping_expr = F.create_map(
    [F.lit(x) for x in chain(*broadcast_dept_names.value.items())]
)

df = emp.withColumn("department_name", mapping_expr[F.col("department_id")])

In [23]:
df.show()

+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+---------------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|department_name|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+---------------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            8|   Department 8|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            7|   Department 7|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900.0|           10|  Department 10|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            1|   Department 1|
|  Michelle|   Elliott|      Air cabin cr

In [26]:
df.where("department_id = 6").groupBy("department_id").agg(sum("salary").cast("long")).show()

+-------------+---------------------------+
|department_id|CAST(sum(salary) AS BIGINT)|
+-------------+---------------------------+
|            6|                50294510721|
+-------------+---------------------------+



#### Accumulators

*  Rule of thumb
Use sum(), count(), avg(), groupBy(), etc. for calculations on your data.
Use accumulators for monitoring, debugging, and counters.

* If you're using an accumulator to compute a result that could be expressed as a DataFrame aggregation, you're usually doing it the non-Spark way.